In [1]:
import os
import torch

os.environ['RWKV_JIT_ON'] = '1'  # Enable JIT compilation for performance

os.environ['RWKV_MY_TESTING'] = 'x070'

os.environ['RWKV_HEAD_SIZE'] = '64'


from src.model import RWKV, RWKV_Tmix_x070

/home/yingte/anaconda3/envs/world/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-08-02 15:03:54,142] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/yingte/anaconda3/envs/world/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/yingte/anaconda3/envs/world/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/yingte/anaconda3/envs/world/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/yingte/anaconda3/envs/world/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/yingte/anaconda3/envs/world/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/yingte/anaconda3/envs/world/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/yingte/anaconda3/env

[2025-08-02 15:03:55,363] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
RWKV_MY_TESTING x070


Using /home/yingte/.cache/torch_extensions/py312_cu126 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/yingte/.cache/torch_extensions/py312_cu126/wind_backstepping/build.ninja...
/home/yingte/anaconda3/envs/world/lib/python3.12/site-packages/torch/utils/cpp_extension.py:2356: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wind_backstepping...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


ninja: no work to do.


Loading extension module wind_backstepping...


In [2]:
class Args:
    def __init__(self):
        self.wandb = ""
        self.proj_dir = "outout"
        self.data_file = "data/minipile"
        self.data_type = "binidx"
        self.vocab_size = 65536
        self.my_testing = "x070"
        self.ctx_len = 512
        self.train_stage = 1
        self.epoch_count = 1
        self.epoch_begin = 0
        self.epoch_save = 1
        self.weight_decay = 0
        self.head_size = 64
        self.num_nodes = 1
        self.micro_bsz = 1
        self.n_layer = 12
        self.n_embd = 768
        self.my_exit_tokens = 1498226207
        self.magic_prime = 2926181
        self.lr_init = 1e-5
        self.lr_final = 1e-5
        self.warmup_steps = 10
        self.beta1 = 0.9
        self.beta2 = 0.99
        self.adam_eps = 1e-8
        self.accelerator = "cpu"
        self.devices = 1
        self.precision = "bf16"
        self.strategy = "deepspeed_stage_2"
        self.grad_cp = 1

args = Args()

In [3]:
model = RWKV(args)

In [4]:
model.parameters

<bound method Module.parameters of RWKV(
  (emb): Embedding(65536, 768)
  (blocks): ModuleList(
    (0): Block(
      (ln1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ln0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (att): RWKV_Tmix_x070(
        (time_shift): RecursiveScriptModule(original_name=ZeroPad2d)
        (receptance): RecursiveScriptModule(original_name=Linear)
        (key): RecursiveScriptModule(original_name=Linear)
        (value): RecursiveScriptModule(original_name=Linear)
        (output): RecursiveScriptModule(original_name=Linear)
        (ln_x): RecursiveScriptModule(original_name=GroupNorm)
      )
      (ffn): RWKV_CMix_x070(
        (time_shift): RecursiveScriptModule(original_name=ZeroPad2d)
        (key): RecursiveScriptModule(original_name=Linear)
        (value): RecursiveScriptModule(original_name=Linear)
      )
    )
    (1-11): 11 x Block(
      (ln1): L

In [5]:
model.blocks[0].ln1

LayerNorm((768,), eps=1e-05, elementwise_affine=True)

In [8]:
tmix = RWKV_Tmix_x070(args, 0)

In [9]:
tmix.reset_parameters()

AttributeError: 'RecursiveScriptModule' object has no attribute 'reset_parameters'

In [ ]:
# Load a pre-trained model into model2
model.load_state_dict(
    torch.load('trained/L12-D768-x070/rwkv-0.pth', map_location='cuda')
)

OrderedDict([('emb.weight',
              tensor([[-5.0049e-02,  2.7344e-02, -5.7617e-02,  ..., -3.5553e-03,
                       -1.7944e-02, -7.0801e-02],
                      [ 5.7220e-06, -4.1485e-05, -5.6028e-05,  ...,  2.1100e-05,
                        4.8161e-05,  2.6226e-05],
                      [-4.6539e-04,  4.1008e-05, -9.2387e-07,  ...,  1.1520e-03,
                       -1.2512e-02, -1.2451e-02],
                      ...,
                      [-6.4850e-05,  4.6790e-06, -7.5340e-05,  ...,  7.4863e-05,
                       -6.9141e-05, -5.2452e-05],
                      [-9.8705e-05,  5.6028e-05,  5.4121e-05,  ...,  2.0742e-05,
                        3.6955e-06,  6.0558e-05],
                      [-9.7752e-05,  6.7711e-05, -5.0306e-05,  ...,  6.6280e-05,
                        7.0572e-05,  4.9829e-05]], device='cuda:0', dtype=torch.bfloat16)),
             ('blocks.0.ln1.weight',
              tensor([0.7852, 0.8125, 0.7539, 0.7500, 0.7734, 0.7773, 0.7891, 0.

In [ ]:
with torch.device('cuda'):
    xx = torch.randn(1, 512, 1)

In [11]:
xx

tensor([[[-4.2690e-01],
         [ 1.0015e+00],
         [-1.1380e+00],
         [-4.9224e-01],
         [-6.2751e-01],
         [ 6.3252e-01],
         [ 2.6693e-01],
         [ 2.1303e-01],
         [-1.4863e+00],
         [ 1.2306e+00],
         [-8.6197e-01],
         [-1.9726e+00],
         [-5.9114e-02],
         [ 1.0889e+00],
         [-1.9060e-01],
         [ 1.1328e+00],
         [-1.2847e-01],
         [-1.1512e+00],
         [ 1.4985e+00],
         [-1.7419e-02],
         [ 5.1417e-01],
         [ 1.8407e+00],
         [ 1.4178e+00],
         [ 1.4596e+00],
         [-2.9609e-01],
         [-6.4911e-01],
         [ 1.3159e+00],
         [ 1.0790e+00],
         [ 5.1325e-01],
         [-1.1154e+00],
         [ 1.3783e+00],
         [-8.3740e-02],
         [-6.4483e-01],
         [ 3.4626e-01],
         [-2.1086e-01],
         [ 2.5847e-01],
         [-1.1798e+00],
         [ 9.4968e-01],
         [-1.2486e+00],
         [-3.9897e-01],
         [ 2.4949e+00],
         [ 8.386